In [64]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup

In [65]:
url = "https://cine21.com/"
response = requests.get(url)
response

<Response [200]>

In [66]:
print(response.text)

<!DOCTYPE html>
<html lang="ko">
<head>
    <!-- Google tag (gtag.js) -->
    <script async src="https://www.googletagmanager.com/gtag/js?id=G-1VP0QP407P"></script>
    <script>
        window.dataLayer = window.dataLayer || [];
        function gtag(){dataLayer.push(arguments);}
        gtag('js', new Date());
        gtag('config', 'G-1VP0QP407P');
    </script>
    <title>씨네21</title>
    <meta charset="utf-8"/>
    <meta http-equiv="X-UA-Compatible" content="IE=edge"/>
    <meta http-equiv="Content-Type" content="text/html; charset=utf-8"/>
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta name="format-detection" content="telephone=no, address=no, email=no"/>
            <meta property="description" content="대한민국 최고 영화전문매체 씨네21은 최신영화 정보, 전문가 평점, 박스오피스 영화, 시사회 이벤트 정보 등 최다 영화 관련 기사와 정보를 제공합니다." />
            <meta property="og:type" content="website" />
            <meta property="og:title" content="씨네21" />
            <meta property="og:descripti

In [67]:
"list_with_upthumb_item" in response.text

True

In [68]:
soup = BeautifulSoup(response.text, "html.parser")

In [69]:
movie_list = soup.select(
    "div.wide_gray_bg_wrap_inner div.swiper-wrapper div.list_with_upthumb_item"
)

movies = []
linkes = []

for movie in movie_list:
    title_tag = movie.select_one("p.title")
    rank_tag = movie.select_one("div.rank")
    score_tag = movie.select_one("p.num")
    info = movie.select("div.etc_info p")
    link_tag = movie.select_one("a")

    title = title_tag.get_text(strip=True) if title_tag else None
    rank = rank_tag.get_text(strip=True) if rank_tag else None
    score = score_tag.get_text(strip=True) if score_tag else None

    daily_audience = info[0].get_text(strip=True) if len(info) > 0 else None
    total_audience = info[1].get_text(strip=True) if len(info) > 1 else None
    open_date = info[2].get_text(strip=True) if len(info) > 2 else None

    movie_page = link_tag.get("href") if link_tag else None

    movie_data = {
        "title": title,
        "rank": rank,
        "score": score,
        "daily_aud": daily_audience,
        "total_aud": total_audience,
        "open_date": open_date,
        "link": movie_page
    }

    linkes.append(movie_page)
    movies.append(movie_data)


In [70]:
movies

[{'title': '스파이더맨: 브랜드 뉴 데이',
  'rank': '1',
  'score': '7.33',
  'daily_aud': '2,124,592명',
  'total_aud': '누적 : 5,910,553명',
  'open_date': '개봉일 : 2026-07-29',
  'link': '/movie/info/?movie_id=63091'},
 {'title': '오디세이',
  'rank': '2',
  'score': '7.55',
  'daily_aud': '1,871,575명',
  'total_aud': '누적 : 1,872,877명',
  'open_date': '개봉일 : 2026-08-05',
  'link': '/movie/info/?movie_id=62585'},
 {'title': '사랑의 하츄핑: 고래보석의 전설',
  'rank': '3',
  'score': '7.00',
  'daily_aud': '407,067명',
  'total_aud': '누적 : 433,565명',
  'open_date': '개봉일 : 2026-08-05',
  'link': '/movie/info/?movie_id=63075'},
 {'title': '호프',
  'rank': '4',
  'score': '6.83',
  'daily_aud': '219,132명',
  'total_aud': '누적 : 4,393,518명',
  'open_date': '개봉일 : 2026-07-15',
  'link': '/movie/info/?movie_id=62480'},
 {'title': '미니언즈 & 몬스터즈',
  'rank': '5',
  'score': '6.50',
  'daily_aud': '36,881명',
  'total_aud': '누적 : 763,404명',
  'open_date': '개봉일 : 2026-07-15',
  'link': '/movie/info/?movie_id=63032'},
 {'title': '다윗',


In [71]:
linkes

['/movie/info/?movie_id=63091',
 '/movie/info/?movie_id=62585',
 '/movie/info/?movie_id=63075',
 '/movie/info/?movie_id=62480',
 '/movie/info/?movie_id=63032',
 '/movie/info/?movie_id=63108',
 '/movie/info/?movie_id=63282',
 '/movie/info/?movie_id=62799',
 '/movie/info/?movie_id=62818',
 'javascript:;']

In [72]:
link_pages = []

for link in linkes:

    if link.startswith("/movie/info/"):
        link_pages.append(link)
    else:
        link_pages.append(None)

link_pages

['/movie/info/?movie_id=63091',
 '/movie/info/?movie_id=62585',
 '/movie/info/?movie_id=63075',
 '/movie/info/?movie_id=62480',
 '/movie/info/?movie_id=63032',
 '/movie/info/?movie_id=63108',
 '/movie/info/?movie_id=63282',
 '/movie/info/?movie_id=62799',
 '/movie/info/?movie_id=62818',
 None]

In [73]:
wanted = ["장르", "등급", "시간", "국가", "감독"]

movie_info_list = []

for link in link_pages:

    # 상세페이지가 없는 영화
    if link is None:
        movie_info_list.append({})
        continue

    # 1. 영화 한 편의 상세페이지 요청
    url = "https://cine21.com" + link
    response1 = requests.get(url)

    soup1 = BeautifulSoup(response1.text, "html.parser")

    # 2. 상세정보 가져오기
    info_list = soup1.select("ul.info_list li")

    # 3. 영화 한 편의 정보를 저장할 딕셔너리
    movie_info = {}

    # 4. 원하는 정보만 수집
    for info in info_list:

        title_tag = info.select_one("p.title")

        if title_tag:
            title = title_tag.get_text(strip=True)

            if title in wanted:
                full_text = info.get_text(" ", strip=True)
                value = full_text.replace(title, "", 1).strip()

                movie_info[title] = value

    # 영화 한 편 수집 완료
    movie_info_list.append(movie_info)

movie_info_list

[{'등급': '12세이상관람가',
  '시간': '144분',
  '장르': '판타지, 어드벤처, 액션',
  '국가': '미국',
  '감독': '데스틴 크리튼'},
 {'등급': '15세이상관람가',
  '시간': '172분',
  '장르': '액션, 모험, 드라마',
  '국가': '미국',
  '감독': '크리스토퍼 놀란'},
 {'등급': '전체 관람가', '시간': '105분', '장르': '애니메이션', '국가': '한국', '감독': '김수훈'},
 {'등급': '15세이상관람가',
  '시간': '156분',
  '장르': '액션, 스릴러, SF',
  '국가': '한국',
  '감독': '나홍진'},
 {'등급': '전체 관람가', '시간': '89분', '장르': '애니메이션', '국가': '미국', '감독': '피에르 코팽'},
 {'등급': '전체 관람가',
  '시간': '109분',
  '장르': '애니메이션',
  '국가': '미국',
  '감독': '필 커닝햄 브렌트 도스'},
 {'등급': '12세이상관람가', '시간': '101분', '장르': '다큐멘터리', '국가': '일본', '감독': '후지노 토모아키'},
 {'등급': '전체 관람가',
  '시간': '101분',
  '장르': '애니메이션',
  '국가': '미국',
  '감독': '앤드류 스탠튼 맥케나 해리스'},
 {'등급': '전체 관람가', '시간': '115분', '장르': '어드벤처, 액션', '국가': '미국', '감독': '토마스 케일'},
 {}]

In [74]:
movie_review_list = []

for link in link_pages:

    # 상세페이지 자체가 없는 영화
    if link is None:
        movie_review_list.append(None)
        continue

    url = "https://cine21.com" + link
    response2 = requests.get(url)

    soup2 = BeautifulSoup(response2.text, "html.parser")

    review_list = soup2.select("ul.expert_star_list li")

    # 영화 한 편의 리뷰
    movie_review = []

    for review in review_list:

        review_text = review.select_one("div.review")

        if review_text:
            review_20 = review_text.get_text(strip=True)
            movie_review.append(review_20)

    # 영화 한 편 수집 완료
    movie_review_list.append(movie_review)

movie_review_list

[['하이틴의 겉옷을 벗고 근본으로 돌아간 성장',
  '큰 힘과 큰 책임이 서로를 따르지 못할 때에도 당신이 히어로라면',
  '도파민형 영웅이 가득한 세상에서 ‘선함’을 지켜낸 ‘강한’ 히어로',
  '역시 스파이더맨은 고통받을수록 강하고 재밌다',
  '캐릭터도 캐스트도 발맞춰 잘 자랐구나',
  '역대 가장 성숙한 스파이더맨의 탄생',
  '쑥 성장한 캐릭터, 따뜻하게 감싸는 드라마',
  '샘 레이미의 유산마저 물려받아 진화한 멜로드라마와 역동의 좋은 균형',
  '아마도 마블의 새로운 미래'],
 ['그리스신화는 놀런을 만나길 오랫동안 기다려왔구나',
  '늘 해오던 걸 가장 잘 할 수 있는 방식으로',
  '죽음과 죄책감을 등에 업은 영웅의 서사',
  '좋으나 싫으나 미우나 고우나 크리스토퍼 놀런의 여름 블록버스터',
  '원전과 어긋나는 놀런의 속도, 이 어긋남을 적절히 조율하려는 음악',
  '야심가다운 변용과 뚝심의 산문시',
  '거대한 함선을 타고 신화와 영화 사이를 전속으로 항해하다',
  '놀런, 인간의 탈을 쓴 영화의 신',
  '어마어마한 게 꼭 좋은 건 아니지만 적어도, 어설프게 어마어마한 것들을 가려내는 역할은 한다',
  'VFX와 핍진성 사이에서 펼쳐진 신화적 풍경',
  '귀환이라는 테마의 원형, <오디세이아>의 <오펜하이머>적 각색'],
 ['시네마 키드는 어떻게 육성되는가, 이 영화를 보고 자랄 어린이들에게'],
 ['후진 없는 희망의 트럭',
  '쾌감의 짝인 공허까지도 납득시킨다',
  '끌렸다, 즐겼다, 낚였다',
  '순식간에 펼쳐지는 아수라, 서스펜스, 페이소스',
  'Grosse Fatigue: 하루가 참 길다',
  '공백을 만들고, 응시하고, 추격한다 계속!'],
 ['직업 체험도 좋지만 역시 본업 할 때가 최고', '천사들의 도시로 간 헨리 제임스, 이야기 대신 영화사를 쓰다'],
 ['검 대신 악기를 드는 성경 뮤지컬'],
 ['‘무엇을 찍었나’로 시작되는 고통. ‘어디서 끊을까’로 확장되는 고뇌'

In [75]:
len(movie_review_list)

10

In [76]:
movie_master_score = []

for link in link_pages:

    # 유효한 상세페이지가 없는 영화
    if not link or not link.startswith("/movie/info/"):
        movie_master_score.append(None)
        continue

    url = "https://cine21.com" + link
    response2 = requests.get(url)

    soup2 = BeautifulSoup(response2.text, "html.parser")

    master_score = soup2.select_one("div.star_wrap")

    # star_wrap 자체가 없는 경우
    if not master_score:
        movie_master_score.append(None)
        continue

    score = master_score.select_one("p.num")

    # 전문가 평점이 없는 경우
    if not score:
        movie_master_score.append(None)
        continue

    m_score = score.get_text(strip=True)
    movie_master_score.append(m_score)

movie_master_score

['7.33', '7.55', '7.00', '6.83', '6.50', '5.00', '6.50', '6.83', '5.67', None]

In [77]:
df = pd.DataFrame(movies)
df

,title,rank,score,daily_aud,total_aud,open_date,link
0,스파이더맨: 브랜드 뉴 데이,1,7.33,"2,124,592명","누적 : 5,910,553명",개봉일 : 2026-07-29,/movie/info/?movie_id=63091
1,오디세이,2,7.55,"1,871,575명","누적 : 1,872,877명",개봉일 : 2026-08-05,/movie/info/?movie_id=62585
2,사랑의 하츄핑: 고래보석의 전설,3,7.00,"407,067명","누적 : 433,565명",개봉일 : 2026-08-05,/movie/info/?movie_id=63075
3,호프,4,6.83,"219,132명","누적 : 4,393,518명",개봉일 : 2026-07-15,/movie/info/?movie_id=62480
4,미니언즈 & 몬스터즈,5,6.50,"36,881명","누적 : 763,404명",개봉일 : 2026-07-15,/movie/info/?movie_id=63032
5,다윗,6,5.00,"22,913명","누적 : 289,372명",개봉일 : 2026-07-10,/movie/info/?movie_id=63108
6,어떻게 해야 했을까?,7,6.50,"22,558명","누적 : 49,713명",개봉일 : 2026-07-29,/movie/info/?movie_id=63282
7,토이 스토리 5,8,6.83,"19,707명","누적 : 2,938,990명",개봉일 : 2026-06-17,/movie/info/?movie_id=62799
8,모아나,9,5.67,"13,996명","누적 : 1,020,487명",개봉일 : 2026-07-08,/movie/info/?movie_id=62818
9,명탐정 코난: 하이웨이의 타천사,10,NaN,"8,828명","누적 : 8,828명",NaN,javascript:;


In [78]:
df1 = pd.DataFrame(movie_info_list)
df1

,등급,시간,장르,국가,감독
0,12세이상관람가,144분,"판타지, 어드벤처, 액션",미국,데스틴 크리튼
1,15세이상관람가,172분,"액션, 모험, 드라마",미국,크리스토퍼 놀란
2,전체 관람가,105분,애니메이션,한국,김수훈
3,15세이상관람가,156분,"액션, 스릴러, SF",한국,나홍진
4,전체 관람가,89분,애니메이션,미국,피에르 코팽
5,전체 관람가,109분,애니메이션,미국,필 커닝햄 브렌트 도스
6,12세이상관람가,101분,다큐멘터리,일본,후지노 토모아키
7,전체 관람가,101분,애니메이션,미국,앤드류 스탠튼 맥케나 해리스
8,전체 관람가,115분,"어드벤처, 액션",미국,토마스 케일
9,NaN,NaN,NaN,NaN,NaN


In [79]:
df3 = pd.DataFrame(movie_master_score)
df3

,0
0,7.33
1,7.55
2,7.00
3,6.83
4,6.50
5,5.00
6,6.50
7,6.83
8,5.67
9,NaN


In [80]:
df4 = pd.DataFrame({
    "review": movie_review_list
})

df4

,review
0,"[하이틴의 겉옷을 벗고 근본으로 돌아간 성장, 큰 힘과 큰 책임이 서로를 따르지 못..."
1,"[그리스신화는 놀런을 만나길 오랫동안 기다려왔구나, 늘 해오던 걸 가장 잘 할 수 ..."
2,"[시네마 키드는 어떻게 육성되는가, 이 영화를 보고 자랄 어린이들에게]"
3,"[후진 없는 희망의 트럭, 쾌감의 짝인 공허까지도 납득시킨다, 끌렸다, 즐겼다, 낚..."
4,"[직업 체험도 좋지만 역시 본업 할 때가 최고, 천사들의 도시로 간 헨리 제임스, ..."
5,[검 대신 악기를 드는 성경 뮤지컬]
6,"[‘무엇을 찍었나’로 시작되는 고통. ‘어디서 끊을까’로 확장되는 고뇌, 미해결 과..."
7,"[그 무한한 애정을 어찌 사랑하지 않을 수 있을까, 뭉클하고 눈물겨우나 이젠 ‘레거..."
8,"[붕어빵이 맛보다 비주얼을 뽐내면, 디즈니 실사영화 논란을 잠재울 충실한 구현력, ..."
9,None


In [81]:
df_total = pd.concat(
    [df, df1, df3, df4],
    axis=1
)

In [82]:
df_total

,title,rank,score,daily_aud,total_aud,open_date,link,등급,시간,장르,국가,감독,0,review
0,스파이더맨: 브랜드 뉴 데이,1,7.33,"2,124,592명","누적 : 5,910,553명",개봉일 : 2026-07-29,/movie/info/?movie_id=63091,12세이상관람가,144분,"판타지, 어드벤처, 액션",미국,데스틴 크리튼,7.33,"[하이틴의 겉옷을 벗고 근본으로 돌아간 성장, 큰 힘과 큰 책임이 서로를 따르지 못..."
1,오디세이,2,7.55,"1,871,575명","누적 : 1,872,877명",개봉일 : 2026-08-05,/movie/info/?movie_id=62585,15세이상관람가,172분,"액션, 모험, 드라마",미국,크리스토퍼 놀란,7.55,"[그리스신화는 놀런을 만나길 오랫동안 기다려왔구나, 늘 해오던 걸 가장 잘 할 수 ..."
2,사랑의 하츄핑: 고래보석의 전설,3,7.00,"407,067명","누적 : 433,565명",개봉일 : 2026-08-05,/movie/info/?movie_id=63075,전체 관람가,105분,애니메이션,한국,김수훈,7.00,"[시네마 키드는 어떻게 육성되는가, 이 영화를 보고 자랄 어린이들에게]"
3,호프,4,6.83,"219,132명","누적 : 4,393,518명",개봉일 : 2026-07-15,/movie/info/?movie_id=62480,15세이상관람가,156분,"액션, 스릴러, SF",한국,나홍진,6.83,"[후진 없는 희망의 트럭, 쾌감의 짝인 공허까지도 납득시킨다, 끌렸다, 즐겼다, 낚..."
4,미니언즈 & 몬스터즈,5,6.50,"36,881명","누적 : 763,404명",개봉일 : 2026-07-15,/movie/info/?movie_id=63032,전체 관람가,89분,애니메이션,미국,피에르 코팽,6.50,"[직업 체험도 좋지만 역시 본업 할 때가 최고, 천사들의 도시로 간 헨리 제임스, ..."
5,다윗,6,5.00,"22,913명","누적 : 289,372명",개봉일 : 2026-07-10,/movie/info/?movie_id=63108,전체 관람가,109분,애니메이션,미국,필 커닝햄 브렌트 도스,5.00,[검 대신 악기를 드는 성경 뮤지컬]
6,어떻게 해야 했을까?,7,6.50,"22,558명","누적 : 49,713명",개봉일 : 2026-07-29,/movie/info/?movie_id=63282,12세이상관람가,101분,다큐멘터리,일본,후지노 토모아키,6.50,"[‘무엇을 찍었나’로 시작되는 고통. ‘어디서 끊을까’로 확장되는 고뇌, 미해결 과..."
7,토이 스토리 5,8,6.83,"19,707명","누적 : 2,938,990명",개봉일 : 2026-06-17,/movie/info/?movie_id=62799,전체 관람가,101분,애니메이션,미국,앤드류 스탠튼 맥케나 해리스,6.83,"[그 무한한 애정을 어찌 사랑하지 않을 수 있을까, 뭉클하고 눈물겨우나 이젠 ‘레거..."
8,모아나,9,5.67,"13,996명","누적 : 1,020,487명",개봉일 : 2026-07-08,/movie/info/?movie_id=62818,전체 관람가,115분,"어드벤처, 액션",미국,토마스 케일,5.67,"[붕어빵이 맛보다 비주얼을 뽐내면, 디즈니 실사영화 논란을 잠재울 충실한 구현력, ..."
9,명탐정 코난: 하이웨이의 타천사,10,NaN,"8,828명","누적 : 8,828명",NaN,javascript:;,NaN,NaN,NaN,NaN,NaN,NaN,None


In [83]:
df_total.to_csv(
    "../data/cine21_movies.csv",
    index=False,
    encoding="utf-8-sig"
)

In [84]:
df_total["daily_aud"] = (
    df_total["daily_aud"]
    .str.replace(",", "")
    .str.replace("명", "")
)

# 누적 관객수
df_total["total_aud"] = (
    df_total["total_aud"]
    .str.replace("누적 :", "")
    .str.replace(",", "")
    .str.replace("명", "")
    .str.strip()
)

# 개봉일
df_total["open_date"] = (
    df_total["open_date"]
    .str.replace("개봉일 :", "")
    .str.strip()
)

df_total["등급"] = (
    df_total["등급"]
    .str.replace("관람가", "")
    .str.strip())

df_total["시간"] = (
    df_total["시간"]
    .str.replace("분", "")
    .str.strip())

df_total["daily_aud"] = pd.to_numeric(df_total["daily_aud"], errors="coerce")
df_total["total_aud"] = pd.to_numeric(df_total["total_aud"], errors="coerce")
df_total["시간"] = pd.to_numeric(df_total["시간"], errors="coerce")

df_total["open_date"] = pd.to_datetime(df_total["open_date"], errors="coerce")

In [85]:
df_total

,title,rank,score,daily_aud,total_aud,open_date,link,등급,시간,장르,국가,감독,0,review
0,스파이더맨: 브랜드 뉴 데이,1,7.33,2124592,5910553,2026-07-29,/movie/info/?movie_id=63091,12세이상,144.0,"판타지, 어드벤처, 액션",미국,데스틴 크리튼,7.33,"[하이틴의 겉옷을 벗고 근본으로 돌아간 성장, 큰 힘과 큰 책임이 서로를 따르지 못..."
1,오디세이,2,7.55,1871575,1872877,2026-08-05,/movie/info/?movie_id=62585,15세이상,172.0,"액션, 모험, 드라마",미국,크리스토퍼 놀란,7.55,"[그리스신화는 놀런을 만나길 오랫동안 기다려왔구나, 늘 해오던 걸 가장 잘 할 수 ..."
2,사랑의 하츄핑: 고래보석의 전설,3,7.00,407067,433565,2026-08-05,/movie/info/?movie_id=63075,전체,105.0,애니메이션,한국,김수훈,7.00,"[시네마 키드는 어떻게 육성되는가, 이 영화를 보고 자랄 어린이들에게]"
3,호프,4,6.83,219132,4393518,2026-07-15,/movie/info/?movie_id=62480,15세이상,156.0,"액션, 스릴러, SF",한국,나홍진,6.83,"[후진 없는 희망의 트럭, 쾌감의 짝인 공허까지도 납득시킨다, 끌렸다, 즐겼다, 낚..."
4,미니언즈 & 몬스터즈,5,6.50,36881,763404,2026-07-15,/movie/info/?movie_id=63032,전체,89.0,애니메이션,미국,피에르 코팽,6.50,"[직업 체험도 좋지만 역시 본업 할 때가 최고, 천사들의 도시로 간 헨리 제임스, ..."
5,다윗,6,5.00,22913,289372,2026-07-10,/movie/info/?movie_id=63108,전체,109.0,애니메이션,미국,필 커닝햄 브렌트 도스,5.00,[검 대신 악기를 드는 성경 뮤지컬]
6,어떻게 해야 했을까?,7,6.50,22558,49713,2026-07-29,/movie/info/?movie_id=63282,12세이상,101.0,다큐멘터리,일본,후지노 토모아키,6.50,"[‘무엇을 찍었나’로 시작되는 고통. ‘어디서 끊을까’로 확장되는 고뇌, 미해결 과..."
7,토이 스토리 5,8,6.83,19707,2938990,2026-06-17,/movie/info/?movie_id=62799,전체,101.0,애니메이션,미국,앤드류 스탠튼 맥케나 해리스,6.83,"[그 무한한 애정을 어찌 사랑하지 않을 수 있을까, 뭉클하고 눈물겨우나 이젠 ‘레거..."
8,모아나,9,5.67,13996,1020487,2026-07-08,/movie/info/?movie_id=62818,전체,115.0,"어드벤처, 액션",미국,토마스 케일,5.67,"[붕어빵이 맛보다 비주얼을 뽐내면, 디즈니 실사영화 논란을 잠재울 충실한 구현력, ..."
9,명탐정 코난: 하이웨이의 타천사,10,NaN,8828,8828,NaT,javascript:;,NaN,NaN,NaN,NaN,NaN,NaN,None


In [86]:
df_total.to_csv(
    "../data/cine21_movies.csv",
    index=False,
    encoding="utf-8-sig"
)